# 🎬 VibeMV CogVideoX: Text-to-Video Generation

## 🚀 Direct Text-to-Video

**Optimized for T4:** Uses 4-bit quantization to fit the massive 20GB+ model into 15GB VRAM.


In [ ]:
# @title ✅ Check GPU
import torch

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {vram:.1f} GB')
else:
    raise SystemExit('❌ No GPU!')

In [ ]:
# @title 📦 Install CogVideoX (5 minutes)
%%capture

!pip install -q torch torchvision
!pip install -q diffusers==0.30.0 transformers accelerate
!pip install -q imageio imageio-ffmpeg opencv-python pillow
!pip install -q bitsandbytes sentencepiece protobuf
!pip install -q moviepy

print('✅ CogVideoX ready!')

In [ ]:
# @title 📤 Upload Timeline
from google.colab import files
import json

uploaded = files.upload()
with open(list(uploaded.keys())[0], 'r') as f:
    timeline = json.load(f)

print(f"✅ {len(timeline['scenes'])} scenes loaded")

In [ ]:
# @title 🎥 Generate Videos with CogVideoX
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
import torch
import os

os.makedirs('cogvideo_clips', exist_ok=True)

print('Loading CogVideoX-2B (T4 Optimized)...')

# Use 4-bit quantization to fit EVERYTHING in VRAM
from transformers import T5EncoderModel, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print('   Loading Text Encoder (4-bit)...')
text_encoder = T5EncoderModel.from_pretrained(
    "THUDM/CogVideoX-2b",
    subfolder="text_encoder",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
)

print('   Loading Pipeline...')
pipe = CogVideoXPipeline.from_pretrained(
    'THUDM/CogVideoX-2b',
    text_encoder=text_encoder,
    torch_dtype=torch.float16
)

# ❌ DISABLED: Cannot offload quantized models
# pipe.enable_model_cpu_offload()

# Manual device placement for non-quantized parts
pipe.transformer.to('cuda')
pipe.vae.to('cuda')

# ✅ ENABLED: VAE optimizations
if hasattr(pipe, 'enable_vae_slicing'):
    pipe.enable_vae_slicing()
if hasattr(pipe, 'enable_vae_tiling'):
    pipe.enable_vae_tiling()

clips = []
failed_scenes = []
print(f"\n🎬 Generating {len(timeline['scenes'])} video clips...\n")

for i, scene in enumerate(timeline['scenes']):
    prompt = scene.get('video_prompt', scene.get('prompt', scene.get('description', '')))
    duration = scene.get('duration', 4.0)
    
    enhanced_prompt = f"{prompt}, cinematic, smooth motion, high quality, 4k"
    print(f"Scene {i+1}: {prompt[:70]}...")
    
    try:
        video = pipe(
            prompt=enhanced_prompt,
            num_frames=40,  # ~1.67 sec at 24fps (must be divisible by 8 and <48)
            num_inference_steps=30,
            guidance_scale=6.0,
            generator=torch.Generator().manual_seed(i)
        ).frames[0]
        
        clip_path = f"cogvideo_clips/scene_{i:03d}.mp4"
        export_to_video(video, clip_path, fps=24)
        clips.append({'path': clip_path, 'duration': duration})
        
        print(f"  ✅ Saved clip\n")
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"  ❌ Error generating scene {i+1}: {str(e)}")
        failed_scenes.append(i+1)
        torch.cuda.empty_cache()

del pipe
del text_encoder
torch.cuda.empty_cache()

if not clips:
    print("\n⚠️ WARNING: No clips were successfully generated!")
    print("   This may be due to:")
    print("   - VRAM limitations (T4 has ~15GB)")
    print("   - Model loading issues")
    print("   - Invalid prompts")
elif failed_scenes:
    print(f"\n⚠️ Generated {len(clips)}/{len(timeline['scenes'])} clips")
    print(f"   Failed scenes: {failed_scenes}")
else:
    print(f"\n✅ Generated {len(clips)} clips!")

In [ ]:
# @title 🎬 Assemble Final Video
try:
    from moviepy.editor import VideoFileClip, concatenate_videoclips
except ImportError:
    print("❌ moviepy not installed! Run the installation cell first.")
    raise

if not clips:
    print("❌ No clips generated!")
else:
    print(f"Stitching {len(clips)} clips...\n")
    video_clips = []
    
    for i, clip in enumerate(clips):
        try:
            print(f"  Loading clip {i+1}/{len(clips)}...")
            vc = VideoFileClip(clip['path'])
            
            # Adjust duration if needed (loop or trim)
            if vc.duration < clip['duration']:
                # Loop if too short
                vc = vc.loop(duration=clip['duration'])
            elif vc.duration > clip['duration']:
                # Trim if too long
                vc = vc.subclip(0, clip['duration'])
            
            video_clips.append(vc)
        except Exception as e:
            print(f"  ⚠️ Error loading clip {i+1}: {e}")
            # Skip problematic clips
            continue
    
    if video_clips:
        print("\nConcatenating clips...")
        final = concatenate_videoclips(video_clips, method="compose")
        
        print("Writing final video...")
        final.write_videofile(
            'vibemv_cogvideo.mp4', 
            fps=24,
            codec='libx264',
            audio_codec='aac'
        )
        
        # Cleanup
        for vc in video_clips:
            vc.close()
        final.close()
        
        print('\n✅ CogVideoX MV complete!')
        files.download('vibemv_cogvideo.mp4')
    else:
        print("❌ No valid clips to stitch!")